# Agent training

In [ ]:
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.env_util import make_vec_env
from gymtest import CircleEnv


# Instantiate the env
# vec_env = make_vec_env(CircleEnv, n_envs=1, env_kwargs=dict())

env = CircleEnv(render_mode="human", log_level="info", vehicles_to_spawn=5)
# env = CircleEnv(render_mode=None, log_level="info", vehicles_to_spawn=2)

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback

class StepLoggerCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(StepLoggerCallback, self).__init__(verbose)
    
    def _on_step(self) -> bool:
        self.logger.record("current_step", self.num_timesteps)
        self.logger.dump(self.num_timesteps)
        return True


# Train the agent
model = A2C("MultiInputPolicy", env, verbose=1, tensorboard_log="./a2c_circle_tensorboard/")
learning_steps = 10000
model.learn(learning_steps, callback=StepLoggerCallback(), tb_log_name="a2c_circle_demo5Vehicles")

In [ ]:
# Quick evaluation
from stable_baselines3.common.evaluation import evaluate_policy
print("Training finished. Starting evaluation")
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=1)
print(mean_reward)
print(std_reward)

In [ ]:
# Execute the trained agent
from tqdm import trange

num_steps = 2
obs, info = env.reset()
for t in range(num_steps):
        actions, _ = model.predict(obs, state=None, deterministic=False)
        observation, reward, terminated, truncated, info = env.step(actions, log_level="debug")

In [ ]:
# Cleanup
env.close()

---
# Test charging stop removal

In [ ]:
from circletest import Simulation

cs_id = "cs_0"
vehicle_id = "myVehicle0"
simulation = Simulation(gui=False)
simulation.add_vehicles()

def print_stops():
    stops = simulation.get_stops(vehicle_id)
    print(f"Stops: {stops}")

# starten
simulation.step()
print_stops()
# rerouten
print("REROUTE")
simulation.reroute_for_charging(vehicle_id, cs_id)
print_stops()
# stop removen
print("REMOVE STOP")
simulation.remove_charging_stop(vehicle_id)
print_stops()

simulation.close()